<a href="https://colab.research.google.com/github/psh-computer/Machine-Learning/blob/main/coursework/language-modeling/day02_lab_minigpt_student_v2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day02 - GPT 구조 이해 및 Next Token Prediction 실습

## 학습 목표

1. GPT 아키텍처 흐름(embedding --> causal self-attention --> FFN -> output)을 단계별로 구현한다.
2. **Causal Mask**가 왜 필요한지 이해하고, `torch.tril`로 직접 만들어본다. (**빈칸 1+2**)
3. **Next Token Prediction(NTP) 학습 루프**의 shift labels 패턴을 직접 작성해 체득한다.(**빈칸3**)
4. temperature/top-k 샘플링으로 텍스트를 생성하고, 파라미터 변화 효과를 확인한다.
5.(Track-B)GPT2-small 사전학습 모델과 Perplexity를 비교해 ""**스케일이 성능을 만든다**"는 사실을 수치로 확인한다.

## 사용 기술
- PyTorch, CUDA(fp16 AMP)
- Hugging Face `datasets`, `transformers`
-TinyStories 데이터셋, gpt2 BPE 토크나이저


In [ ]:
# 패키지 설치 (Colab 최초 1회)
# !pip -q install -U datasets transformers
! pip -q install \
    "datasets==3.6.0" \
    "transformers==4.51.3" \
    "accelerate==1.13.0" \
    "tokenizers==0.21.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 

In [ ]:
# 한글 폰트
!apt-get install -y fonts-nanum > /dev/null 2>&1

## 세션 다시 시작하기 (설치한 패키지 적용)

In [ ]:
from torch.cuda import is_available
import sys, os, math, time, gc
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams['axes.unicode_minus'] = False

print('Python :', sys.version.split()[0])
print("torch  :",torch.__version__)
print('cuda   :', torch.cuda.is_available())
if torch.cuda.is_available():
  print('gpu   :', torch.cuda.get_device_name(0))
  free, total = torch.cuda.mem_get_info()
  print(f"vram : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
  # TF32 : Ampere 이상 GPU에서 matmul 속도 향상 (정밀도 소폭 감소, 학습엔 충분)
  torch.backends.cuda.matmul.allow_tf32 = True
  torch.backends.cudnn.allow_tf32 = True

Python : 3.13.15
torch  : 2.11.0+cu128
cuda   : True
gpu   : Tesla T4
vram : 15.5 GB free / 15.6 GB total


In [ ]:
# CUDA 메모리 단편화 완화 : 큰 텐서 할당/해제 반복 시 OOM 방지
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def seed_everything(seed=42):
  ''' 재현성을 위해 모든 난수 생성기를 동일 시드로 고정한다. '''
  import random, numpy as np
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## 0.[조건부] 토크나이저 이해
** Day01에서 `week01_text_data_tokenizer` 실습을 완료한 경우 Cell 4-5는 건너뜁니다 **

GPT 계열 모델은 텍스트를 **토큰 ID 시퀀스**로 변환해서 입력받습니다.

gpt2 BPE 토크나이저의 동작을 시각화하고, 특수 토큰 규격의 중요성을 확인합니다.

### 토크나이저의 의미

컴퓨터는 '사과'라는 글자를 모릅니다. 토크나이저는 글자를 숫자(ID)로 바꾸는 번역기입니다.

 분절(Tokenization) : "나는 학교에 간다." --> ["나","는","학교","에","간다"]

수치화(Encoding): ["나","는", ...] --> [120, 45, 892, ...]

어휘 사전(Vocab): 모델이 배운 모든 단어와 숫자의 매핑 테이블입니다. (예: 120번은 항상 "나")

In [ ]:
# Section 0-1: gpt2 토크나이저 로드 및 토큰화 시각화
from transformers import AutoTokenizer

_tok = AutoTokenizer.from_pretrained('gpt2')
_tok.pad_token = _tok.eos_token  # GPT2는 pad_token이 없음 -> eos로 대체

_sample = 'Once upon a time, a little girl found a magical key.'
_ids = _tok.encode(_sample)
_tokens = _tok.convert_ids_to_tokens(_ids)

print(f'원문   : {_sample}')
print(f'토큰 수: {len(_ids)}')
print()
print(f'{'토큰 ID':>8} 토큰 문자열')
print('-' * 30)
for tid, tok in zip(_ids, _tokens):
  # G(U+120) : 공백을 다음 단어 앞에 붙여 표시하는 ByteLevel 방식
  # --> decode 시 자동으로 공백으로 복원됨
  print(f'{tid:>8} {tok}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

원문   : Once upon a time, a little girl found a magical key.
토큰 수: 13

   토큰 ID 토큰 문자열
------------------------------
    7454 Once
    2402 Ġupon
     257 Ġa
     640 Ġtime
      11 ,
     257 Ġa
    1310 Ġlittle
    2576 Ġgirl
    1043 Ġfound
     257 Ġa
   10883 Ġmagical
    1994 Ġkey
      13 .


In [ ]:
# Selection 0-2: 특수 토큰 확인 + 임베딩 크기 불일치 경고
print(f'vocab_size    : {_tok.vocab_size}')
print(f'eos_token_ie  : {_tok.eos_token_id} ({_tok.eos_token!r})')
print(f'pad_token_id  : {_tok.pad_token_id} ({_tok.pad_token!r})')
print()

# 임베딩 크기 불일치가 실제로 어떤 에러를 발생시키는지 미리 확인
# vocab_size = 1000인 임베딩에 id=50000을 넣으면 범위 초과
print('[경고 재현] Embedding(1000, 64)에 ID=50000을 넣으면?')
try:
  _emb = torch.nn.Embedding(1000, 64)
  _bad = torch.tensor([50000])
  _ = _emb(_bad)
except IndexError as e:
  print(f' -> IndexError: {e}')

print()
print(f'올바른 설정: nn.Embedding({_tok.vocab_size}, dim)')
print('TinyGPT 생성 시 vocab_size = tokenizer.vocab_size 를 정확히 맞춰야 합니다.')
del _tok, _ids, _tokens, _sample, _emb, _bad

vocab_size    : 50257
eos_token_ie  : 50256 ('<|endoftext|>')
pad_token_id  : 50256 ('<|endoftext|>')

[경고 재현] Embedding(1000, 64)에 ID=50000을 넣으면?
 -> IndexError: index out of range in self

올바른 설정: nn.Embedding(50257, dim)
TinyGPT 생성 시 vocab_size = tokenizer.vocab_size 를 정확히 맞춰야 합니다.


## 1. 데이터 파이프라인 - TinyStories

- 데이터셋 : `roneneldan/TinyStories` (HuggingFace)
- 토크나이저 : `gpt2` BPE (vocab_size=50,257)
- block_size : 256
- 학습 블록 수 : 50,000 (약 12.8M 토큰)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 수업 중 수정할 하이퍼파라미터는 이 셀에서만 관리
BLOCK_SIZE = 256    # 한 번에 처리할 토큰 수 (컨텍스트 길이), T4 기준 안전값
N_TRAIN    = 50_000 # 학습 블록 수, OOM 시 20,000로 축소
N_VALID    = 2_000
BATCH_SIZE = 16     # OOM 시 8로 축소

print('TinyStories 로딩 중')
ds = load_dataset('roneneldan/TinyStories')
print(ds)

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

print(f'\n vocab_size : {tokenizer.vocab_size}')
print(f'eos_token     : {tokenizer.eos_token!r} id = {tokenizer.eos_token_id}')

# 샘플 1건 미리 확인
sample_story = ds['train'][0]['text']
print(f'\n[샘플 원문 (앞 200자)\n{sample_story[:200]}]')
print(f' ...\n원문 길이 : {len(sample_story)} chars')

TinyStories 로딩 중


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

 vocab_size : 50257
eos_token     : '<|endoftext|>' id = 50256

[샘플 원문 (앞 200자)
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on]
 ...
원문 길이 : 701 chars


## 1-1 토큰화 + block화

1. 원문 텍스트 토큰화 -> 긴 시퀀스로 연결(concatenate)
2. 고정 길이 `block_size`로 자르기 -> `input_ids`
3. `labels = input_ids` (NTP: 입력과 레이블이 동일한 시퀀스)

### 블록화의 이유

1. 비효율적인 방식 : 짧은 문장에 맞춰 긴 문장을 자르거나, 긴 문장에 맞춰 짧은 문장에 빈 공강([PAD])을 채움 (연산 낭비)
2. 이 코드의 방식(Packing) : 모든 문장을 기차 놀이처럼 이어 붙인 뒤, 모델이 처리할 수 있는 최대 길이(BLOCK_SIZE)로 깍두기 썰듯 자릅니다. 이렇게 하면 모든 학습 데이터가 빈틈없이 꽉 차서 학습 효율이 극대화됩니다.

In [ ]:
import time

# --------------------------------------------
# [최적화] 원문 스토리를 먼저 일부 선택한 뒤
# 토크나이즈 → block화
#
# 목표:
#   N_TRAIN × BLOCK_SIZE 만큼의 토큰 확보
#
# 예:
#   N_TRAIN = 50,000
#   BLOCK_SIZE = 256
#   필요 토큰 = 12.8M
#
# TinyStories 평균 토큰 수를 200으로 가정하면
#   필요 스토리 ≈ 12.8M / 200 = 64,000
#
# SAFETY_FACTOR = 2.0을 적용하여
#   약 128,000개의 story만 처리
# --------------------------------------------

TOKENS_PER_STORY_EST = 200   # 스토리당 평균 토큰 보수적 추정
SAFETY_FACTOR = 2.0          # block 손실 등을 고려한 여유 배수

# 필요한 원문 story 개수 계산
n_train_raw = int(
    N_TRAIN * BLOCK_SIZE / TOKENS_PER_STORY_EST * SAFETY_FACTOR
)

n_valid_raw = int(
    N_VALID * BLOCK_SIZE / TOKENS_PER_STORY_EST * SAFETY_FACTOR
)

print(
    f'원문 선택 수 - '
    f'train: {n_train_raw:,} | '
    f'valid: {n_valid_raw:,}'
)


# --------------------------------------------
# 필요한 만큼의 원문만 선택
# select()는 인덱스 기반 선택이므로 비교적 가벼움
# --------------------------------------------

ds_train_raw = ds['train'].select(
    range(min(n_train_raw, len(ds['train'])))
)

ds_valid_raw = ds['validation'].select(
    range(min(n_valid_raw, len(ds['validation'])))
)


# --------------------------------------------
# Tokenization
# --------------------------------------------

def tokenize_batch(batch):
    """
    배치 단위 토큰화.

    truncation=False:
        긴 스토리도 중간에서 자르지 않고
        전체 토큰을 유지.
    """

    return tokenizer(
        batch['text'],
        return_attention_mask=False,
        truncation=False
    )


# --------------------------------------------
# Token → fixed-size block
# --------------------------------------------

def group_texts(examples):
    """
    여러 스토리의 토큰을 하나로 이어붙인 뒤
    BLOCK_SIZE 단위로 분할.

    예:
        Story A → [1,2,3]
        Story B → [4,5,6,7]
        Story C → [8,9]

        ↓ concatenate

        [1,2,3,4,5,6,7,8,9]

        ↓ BLOCK_SIZE=4

        [1,2,3,4]
        [5,6,7,8]

        마지막 1개 토큰은 버림.

    labels = input_ids
        Causal Language Modeling에서
        다음 토큰 예측 학습에 사용.
    """

    # 모든 example의 토큰을 하나의 긴 sequence로 연결
    concatenated = {
        k: sum(examples[k], [])
        for k in examples.keys()
    }

    # BLOCK_SIZE로 정확하게 나누어지는 부분까지만 사용
    total_length = (
        len(concatenated['input_ids'])
        // BLOCK_SIZE
    ) * BLOCK_SIZE

    # block_size 단위로 분할
    result = {
        k: [
            v[i:i + BLOCK_SIZE]
            for i in range(0, total_length, BLOCK_SIZE)
        ]
        for k, v in concatenated.items()
    }

    # Causal LM:
    # input_ids 자체를 labels로 사용
    result['labels'] = result['input_ids'].copy()

    return result


# --------------------------------------------
# Tokenization + Block화
# --------------------------------------------

stt = time.time()

print('토큰화 + block화 진행 중...')


# Train
tokenized_train = ds_train_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=['text']
)

# Validation
tokenized_valid = ds_valid_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=['text']
)


# Token → BLOCK_SIZE
lm_train = tokenized_train.map(
    group_texts,
    batched=True
)

lm_valid = tokenized_valid.map(
    group_texts,
    batched=True
)


print(
    f'완료 - 소요 시간: '
    f'{time.time() - stt:.1f}초'
)


# --------------------------------------------
# [검증]
# 필요한 block 수가 확보되었는지 확인
# --------------------------------------------

print(
    f'생성된 block 수 - '
    f'train: {len(lm_train):,} | '
    f'valid: {len(lm_valid):,}'
)


assert len(lm_train) >= N_TRAIN, (
    f'train 블록 부족: '
    f'{len(lm_train):,} < {N_TRAIN:,} '
    f'→ SAFETY_FACTOR를 2.5~3.0으로 높이세요'
)

assert len(lm_valid) >= N_VALID, (
    f'valid 블록 부족: '
    f'{len(lm_valid):,} < {N_VALID:,} '
    f'→ SAFETY_FACTOR를 2.5~3.0으로 높이세요'
)


# --------------------------------------------
# 최종적으로 필요한 block 수만 선택
# --------------------------------------------

train_lm = lm_train.select(
    range(N_TRAIN)
)

valid_lm = lm_valid.select(
    range(N_VALID)
)


# --------------------------------------------
# 최종 확인
# --------------------------------------------

print(
    f'train: {len(train_lm):,} blocks | '
    f'valid: {len(valid_lm):,} blocks'
)

print(
    f'각 block 크기: {BLOCK_SIZE} tokens'
)

print(
    f'총 train tokens: '
    f'{len(train_lm) * BLOCK_SIZE:,}'
)

print(
    f'총 valid tokens: '
    f'{len(valid_lm) * BLOCK_SIZE:,}'
)

원문 선택 수 - train: 128,000 | valid: 5,120
토큰화 + block화 진행 중...


Map:   0%|          | 0/128000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5120 [00:00<?, ? examples/s]

Map:   0%|          | 0/128000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5120 [00:00<?, ? examples/s]

완료 - 소요 시간: 250.7초
생성된 block 수 - train: 110,160 | valid: 3,983
train: 50,000 blocks | valid: 2,000 blocks
각 block 크기: 256 tokens
총 train tokens: 12,800,000
총 valid tokens: 512,000


In [ ]:
# --------------------------------------------
# [검증] 데이터 파이프라인이 의도대로 동작하는지 확인
# --------------------------------------------

# 1) 블록 1건의 shape 확인
sample_block = train_lm[0]            # <-- lm_ds['train'] 에서 변경
print('[블록 구조 확인]')
print(f'input_ids 길이 : {len(sample_block['input_ids'])} (== BLOCK_SIZE={BLOCK_SIZE} 이어야 함)')
print(f'labels    길이 : {len(sample_block['labels'])}')

assert len(sample_block['input_ids']) == BLOCK_SIZE, '블록 크기 불일치!'

# 2) input_ids == labels 확인 (NTP 세팅)
assert sample_block['input_ids'] == sample_block['labels'], 'input_ids != labels!'
print('input_ids == labels : check(NTP 설정 정상)')

# 3) 실제 토큰을 decode해서 의미 있는 텍스트인지 확인
decoded = tokenizer.decode(sample_block['input_ids'][:30], skip_special_tokens=True)
print(f'\n [첫 블록 앞 30 토큰 decode] \n {decoded!r}')

# 4) 전체 토큰 수
n_train_tokens = len(train_lm) * BLOCK_SIZE       # <-- lm_ds['train']에서 변경
print(f'\n[데이터 규모]')
print(f'train 블록 수 : {len(train_lm):,}')         # <-- 동일
print(f'train 토큰 수 : {n_train_tokens/1e6:.1f}M tokens')

[블록 구조 확인]
input_ids 길이 : 256 (== BLOCK_SIZE=256 이어야 함)
labels    길이 : 256
input_ids == labels : check(NTP 설정 정상)

 [첫 블록 앞 30 토큰 decode] 
 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily'

[데이터 규모]
train 블록 수 : 50,000
train 토큰 수 : 12.8M tokens


## 2. 모델 구현 -- TinyGPT

```
input_ids(B, 1)
  |
  +-- tok_emb: Embedding(50257, 256)
  +-- pos_emb: Embedding(256,256)
  |
  +-- GPTBlock x 6
          |
          +-- LayerNorm
          +-- CausalSelfAttention <- 빈칸 1, 2
          +-- LayerNorm
          +-- MLP (FFN)
  |
  +-- LayerNorm
  +-- lm_head: Linear(256, 50257) (Weight Tying)
```

빈칸 1 + 2 -- CausalSelfAttention

**빈칸 1**: Q, K ,V를 계산하고 multi-head 형태로 reshape합니다.
```
힌트:
  qkv = self.qkv_proj(x)                    # (B, T, 3*C)
  q, k, v = qkv.split(C, dim=-1)            # 각각 (B, T, C)
  q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, nh, T, hd)
  # k, v 도 동일하게 변환하세요
  ```


**빈칸2**: 미래 토큰을 보지 못하도록 causal mask를 적용합니다.
```
힌트:
  self.mask는 torch.tril(ones) 로 만든 하삼각 행렬입니다.
  mask == 0 인 위치를 float('-inf')로 채우면
  softmax 후 해당 attention weight 가 0이 됩니다.
  --> attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
```

In [ ]:
class CausalSelfAttention(nn.Module):
  def __init__(self, dim, n_heads, block_size, dropout=0.1):
    super().__init__()
    assert dim % n_heads == 0, 'dim은 n_heads의 배수여야 합니다'
    self.n_heads = n_heads
    self.head_dim = dim // n_heads

    self.qkv_proj = nn.Linear(dim, 3 * dim, bias=False)
    self.out_proj = nn.Linear(dim, dim, bias=False)
    self.attn_drop = nn.Dropout(dropout)
    self.res_drop = nn.Dropout(dropout)

    # causal mask : 하삼각행렬 - 현재 + 이전 = 1, 미래 = 0
    # self.mask
    # register_buffer : 파라미터는 아니지만 state_dict에 포함됨 (저장/로드 시 오지)
    # + 학습은 안 되지만, 모델의 일부처럼 관리되어야 하는 텐서
    self.register_buffer(
        'mask',
        torch.tril(torch.ones(block_size, block_size))
              .view(1, 1, block_size, block_size)
    )

  def forward(self, x):
    B, T, C = x.shape # B = batch, T = 시퀀스 길이, C = dim

    # 빈칸 1: Q, K, V 계산 --------------------------------------------------------
    # qkv_proj(x) -> (B, T, 3 * C) -> Q/K/V 로 split
    # 각각 (B, n_heads, T, head_dim) 형태로 reshape
    #
    # TODO : Q, K, V를 계산하고 (B, n_heads, T, head_dim) shape으로 변환하세요.

    qkv = self.qkv_proj(x)
    q, k, v = qkv.split(C, dim=-1)
    q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    # -------------------------------------------------------------------------------

    # Scaled Dot-Product Attention score (완성 제공)
    scale = self.head_dim ** -0.5   # 1/sqrt()
    attn = (q @ k.transpose(-2, -1)) * scale # (B, nh, T, T)

    # 빈칸 2: Causal mask 적용 --------------------------------------------------------
    # mask==0 인 위치를 float('-inf')로 채워 softmax 후 0이 되게 합니다.
    #
    # * .masked_fill(조건, 값): 조건이 True인 위치를 조정한 값으로 채움
    # -------------------------------------------------------------------------------
    # TODO: attn에 causal mask를 적용하세요.

    attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

    # -------------------------------------------------------------------------------

    attn = F.softmax(attn, dim=-1)
    attn = self.attn_drop(attn)

    out = attn @ v
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return self.res_drop(self.out_proj(out))